### Information Retrieval for 10 CH

* CH DE FLEYRIAT / CH BOURG EN BRESSE
* CH BUGEY SUD
* CH DU HAUT BUGEY - GEOVREISSET
* CLINIQUE CONVERT
* HOPITAL PRIVE D AMBERIEU
* CH SAINT-QUENTIN
* CH LAON
* CH SOISSONS
* CH CHAUNY
* CH CHATEAU-THIERRY

In [3]:
!pip install tavily-python

from tavily import TavilyClient
import pandas as pd
import json


tavily_client = TavilyClient(api_key="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")

In [ ]:
# List of the 11 Hospitals (CH)
hospitals = [
    "CH DE FLEYRIAT", "CH BUGEY SUD", "CH DU HAUT BUGEY", 
   # "CLINIQUE CONVERT", "HOPITAL PRIVE D AMBERIEU", "CH SAINT-QUENTIN",
   # "CH LAON", "CH SOISSONS", "CH CHAUNY", "CH CHATEAU-THIERRY", "CH BOURG EN BRESSE"
]

# Refined Search Templates for 2024
# We keep the keywords in French to match the source documents
query_templates = [
    '("{hospital}") "accès régulé" urgences arrêté 2024',
    '("{hospital}") "fermeture temporaire" urgences 2024',
    '("{hospital}") "régulation" service des urgences 2024 site:ars.sante.fr'
]

all_extracted_results = []

for hospital in hospitals:
    print(f"--- Searching data for: {hospital} ---")
    
    for template in query_templates:
        search_query = template.format(hospital=hospital)
        
        try:
            # We use advanced search to get deep relevant results
            # include_raw_content=True is VITAL for your Mistral-7B model to parse dates later
            response = tavily_client.search(
                query=search_query,
                search_depth="basic", 
                max_results=5, # 5 is enough to find the legal decree (Arrêté)
                include_domains=["ars.sante.fr", "prefectures-regions.gouv.fr"],
                include_raw_content=True 
            )
            
            for result in response.get("results", []):
                all_extracted_results.append({
                    "score": result.get("score"),
                    "hospital": hospital,
                    "query": search_query,
                    "title": result.get("title"),
                    "url": result.get("url"),
                    "raw_text": result.get("raw_content") # This is the clean Markdown for your LLM
                })
        except Exception as e:
            print(f"Error searching for {hospital}: {e}")

# Convert to DataFrame for easy handling
df = pd.DataFrame(all_extracted_results)

# Save to JSON for your Onyxia/Mistral pipeline
#df.to_json("hospital_closures_2024_raw.json", orient="records", force_ascii=False, indent=4)

#print(f"\nSearch complete. {len(df)} potential records saved to 'hospital_closures_2024_raw.json'.")

In [ ]:
# 1. FINAL SEARCH RESULTS DISPLAY (Corrected Logic)
print("\n" + "#" * 80)
print("EDCD: FINAL SEARCH RESULTS DISPLAY")
print("#" * 80 + "\n")

# Group results by query to provide a coherent summary
unique_queries = df['query'].unique()

for i, q_text in enumerate(unique_queries):
    print("=" * 80)
    print(f"QUERY {i+1}: {q_text}")
    print("-" * 80)

    # Filter the dataframe for this specific query
    query_results = df[df['query'] == q_text]
    
    for j, (_, row) in enumerate(query_results.iterrows()):
        if row['title'] == "No results":
            print(f"[{j+1}] No relevant documents found.")
        else:
            print(f"[{j+1}] {row['title']}")
            print(f"URL: {row['url']}")
            print(f"Score: {row['score']}")
            # Truncate content for readable console output
            print(f"Snippet: {row['raw_text'][:300]}...") 
        print()

In [ ]:
# 1. Clean Loop Printing
print("\n" + "#" * 80)
print("FINAL SEARCH RESULTS DISPLAY")
print("#" * 80 + "\n")

for i, resp in enumerate(df):
    print("=" * 80)
    print(f"QUERY {i+1}: {resp['query']}")
    print("-" * 80)

    # Check if there are results for this query
    if not resp.get("results"):
        print("No results found for this specific query.")
    
    for j, r in enumerate(resp["results"]):
        print(f"[{j+1}] {r['title']}")
        print(f"URL: {r['url']}")
        print(f"Score: {r['score']}")
        # Using .get('content') or .get('raw_content') depending on search settings
        content_preview = r.get('content', 'No content available')
        print(f"Snippet: {content_preview[:300]}...")  # truncate
        print()

In [ ]:
query = '(CH "Bourg-en-Bresse") AND (urgences OR "service des urgences") AND (fermeture OR "fermeture temporaire" OR "fermeture définitive" OR "régulation" OR "accès régulé" OR arrêté)'

resp = tavily_client.search(
        query=query,
        search_depth="basic",
        max_results=5,
        include_answer=False,
        include_raw_content=False,
        include_domains=["https://ars.sante.fr/"]  # portal + often links out
    )

print(response)

In [ ]:
# Normalizamos a una lista del tipo que quieres
all_results = [{
    "query": query,
    "results": response.get("results", [])
}]

# Print en el formato que pediste
for i, resp in enumerate(all_results):
    print("=" * 80)
    print(f"QUERY {i+1}: {resp['query']}")
    print("-" * 80)

    for j, r in enumerate(resp["results"]):
        print(f"[{j+1}] {r.get('title')}")
        print(f"URL: {r.get('url')}")
        print(f"Score: {r.get('score')}")
        print(f"Snippet: {r.get('content', '')[:300]}")
        print()

In [ ]:
resp = tavily_client.search(
        query = '"Centre Hospitalier de Bourgoin-Jallieu" AND urgences AND (arrêté OR "réguler temporairement" OR "accès régulé" OR "régulation") site:ars.sante.fr',
        search_depth="basic",
        max_results=5,
        include_answer=True,
        include_raw_content=False,
        include_domains=["https://ars.sante.fr/"]  # portal + often links out
    )

print(resp)

{'query': 'In what dates the Centre Hospitalier (CH) de Bourgoin-Jallieu had any temporary or closure or time regulation in their urgences service?', 'response_time': 8.1, 'follow_up_questions': None, 'answer': 'The Centre Hospitalier de Bourgoin-Jallieu had temporary closures in its urgences service in January 2026. Specific dates are not provided. Current data ends in January 2026.', 'images': [], 'results': [{'url': 'https://www.ars.sante.fr', 'title': 'Agence régionale de santé | Agir pour la santé de tous', 'content': '* [Bretagne](//www.bretagne.ars.sante.fr "Bretagne (nouvelle fenêtre)"). * [Corse](//www.corse.ars.sante.fr "Corse (nouvelle fenêtre)"). * [Guadeloupe](//www.guadeloupe.ars.sante.fr "Guadeloupe (nouvelle fenêtre)"). * [Guyane](//www.guyane.ars.sante.fr "Guyane (nouvelle fenêtre)"). * [Hauts-de-France](//www.hauts-de-france.ars.sante.fr "Hauts-de-France (nouvelle fenêtre)"). * [Ile-de-France](//www.iledefrance.ars.sante.fr "Ile-de-France (nouvelle fenêtre)"). * [La R

In [4]:
query = 'Arrêté "Centre Hospitalier de Bourgoin-Jallieu" urgences réguler temporairement'

resp = tavily_client.search(
    query=query,
    search_depth="advanced",
    max_results=10,
    include_answer=True,
    include_raw_content=False,
    include_domains=["ars.sante.fr", "prefectures-regions.gouv.fr"]
)

all_results = [{
    "query": query,
    "results": resp.get("results", [])
}]

In [5]:
print(resp)

{'query': 'Arrêté Centre Hospitalier de Bourgoin-Jallieu urgences réguler temporairement', 'follow_up_questions': None, 'answer': 'The Centre Hospitalier de Bourgoin-Jallieu temporarily regulates access to its emergency department from 20:00 to 08:30 for three months starting August 2, 2024. This regulation aims to manage patient flow and ensure optimal care.', 'images': [], 'results': [{'url': 'https://www.prefectures-regions.gouv.fr/auvergne-rhone-alpes/irecontenu/telechargement/119090/885663/file/5aout2024_recueil-84-2024-234-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "qui le concerne, de l’exécution du présent arrêté qui sera notifié au directeur de l’établissement de santé et publié au recueil des actes administratifs (RAA) de la préfecture de la région Auvergne-Rhône-Alpes. Fait à Lyon, le 2 AOUT 2024 La Directrice Générale de l’Agence Régionale de Santé Auvergne-Rhône-Alpes Cécile COURREGES Arrêté n°2

{'query': 'Arrêté "Centre Hospitalier de Bourgoin-Jallieu" urgences réguler temporairement', 'results': [{'url': 'https://www.prefectures-regions.gouv.fr/irecontenu/telechargement/123675/916331/file/31d%C3%A9cembre2024_n%C2%B0-2_recueil-84-2024-378-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "de la région Auvergne - Rhône-Alpes. Fait à Lyon, le 30 décembre 2024 La directrice générale de l'agence régionale de santé Auvergne-Rhône-Alpes Cécile COURREGES Arrêté n°2024-17-0862 Portant autorisation de réguler temporairement l’accès aux urgences du centre hospitalier de Bourgoin-Jallieu La directrice générale de l’agence régionale de santé Auvergne-Rhône-Alpes Vu le code de la santé publique, notamment ses articles L. 1432-2, L. 6122-1, L. 6122-8, R. 6122-25, R. 6122-41, R. 6123-1 à R. 6123-32-11 ; Vu le décret du 19 avril 2023 portant nomination de madame Cécile COURREGES en qualité de directrice générale de l’agence régionale de santé Auvergne-Rhône-Alpes ; Vu l’arrêté n°2016-0714 du 21 mars 2016 portant renouvellement tacite d’autorisations d’activités de soins de médecine [...] tacite d’autorisations d’activités de soins de médecine d’urgence du centre hospitalier de Bourgoin-Jallieu ; Vu l’arrêté du 2 juillet 2024 relatif à la régulation temporaire de l’accès aux urgences ; Vu la décision 2024-23-0064 en date du 03 décembre 2024 portant délégation de signature de la Directrice générale de l’ARS Auvergne-Rhône-Alpes ; Vu l’avis consultatif n°2024-15 du 23 décembre 2024 de la section Urgences chargée d'émettre un avis pour les activités de médecine d'urgence du comité consultatif d'allocation des ressources prévu à l'article R. 162-29 du code de la sécurité sociale ; Vu la demande de l’établissement de bénéficier d’un renouvellement de l’autorisation de réguler temporairement la nuit l’accès aux urgences de son territoire ; Considérant que tout établissement de [...] mis en œuvre par le centre hospitalier, l’établissement ne parvient pas à réunir les effectifs nécessaires à l’accueil des urgences sans régulation préalable ; Considérant que, dans ce contexte il y a lieu de prioriser l’accueil des patients au sein de la structure des urgences, de préserver les capacités optimales de prise en charge des urgences vitales et graves des structures mobiles d’urgence et de réanimation, d'assurer une sécurité des soins et d'éviter la saturation des urgences ; ARRÊTE Article 1er : Le présent arrêté prend effet pour 3 mois à compter de sa date de signature ; le centre hospitalier de Bourgoin-Jallieu est autorisé à réguler l’accès à sa structure des urgences entre 20h00 et 8h30. Article 2 : L’accès à la structure des urgences s’opérera par une régulation", 'score': 0.92442214, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/auvergne-rhone-alpes/irecontenu/telechargement/119090/885663/file/5aout2024_recueil-84-2024-234-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "qui le concerne, de l’exécution du présent arrêté qui sera notifié au directeur de l’établissement de santé et publié au recueil des actes administratifs (RAA) de la préfecture de la région Auvergne-Rhône-Alpes. Fait à Lyon, le 2 AOUT 2024 La Directrice Générale de l’Agence Régionale de Santé Auvergne-Rhône-Alpes Cécile COURREGES Arrêté n°2024-17-0264 Portant autorisation de réguler temporairement l’accès aux urgences du Centre Hospitalier Yves TOURAINE La Directrice générale de l’Agence Régionale de Santé d’Auvergne-Rhône-Alpes Vu le code de la santé publique, notamment ses articles L. 1432-2, L. 6122-1, L. 6122-8, R. 6122-25, R. 6122-41, R. 6123-1 à R. 6123-32-11 ; Vu l’arrêté du 2 juillet 2024 relatif à la régulation temporaire de l’accès aux urgences ; Vu le courrier du 21 mars 2016 [...] de la structure des urgences, de préserver les capacités optimales de prise en charge des urgences vitales et graves des structures mobiles d’urgence et de réanimation, d'assurer une sécurité des soins et d'éviter la saturation des urgences ; ARRÊTE Article 1er : Le présent arrêté prend effet pour 3 mois à compter de sa date de signature, le Centre Hospitalier de Bourgoin-Jallieu est autorisé à réguler l’accès à sa structure des urgences entre 20h00 et 8h30. Article 2 : L’accès à la structure des urgences s’opérera par une régulation préalable après appel au SAMU-Centre 15. L'organisation mise en œuvre à l'entrée de la structure des urgences concernée comporte un accueil physique par un professionnel de santé ou par personne titulaire de l’attestation de formation aux gestes et soins [...] (ars-ara-dpd@ars.sante.fr). Conformément au règlement (UE) 2016/679 du Parlement européen et à la loi n° 78-17 du 6 janvier 1978 modifiée relative à l'informatique, aux fichiers et aux libertés, vous pouvez accéder aux données vous concernant ou demander leur effacement. Vous disposez également d'un droit d’opposition, d’un droit de rectification et d’un droit à la limitation du traitement de vos données. Pour exercer ces droits, vous pouvez contacter le Délégué à la protection des données de l’ARS (ars-ara-dpd@ars.sante.fr). Arrêté n°2024-17-0263 Portant autorisation de réguler temporairement l’accès aux urgences du Centre Hospitalier de Bourgoin-Jallieu La Directrice générale de l’Agence Régionale de Santé d’Auvergne-Rhône-Alpes Vu le code de la santé publique, notamment ses articles L.", 'score': 0.8993429, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/irecontenu/telechargement/119139/885999/file/6aout2024_recueil-84-2024-235-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "par le site Internet www.telerecours.fr. Article 8 : Le Directeur de l’offre de soins de l'Agence régionale de santé Auvergne-Rhône-Alpes et le directeur de l’établissement sont chargés, chacun en ce qui les concerne, de l'exécution du présent arrêté, qui sera publié au Recueil des actes administratifs de la Préfecture de la Région Auvergne-Rhône-Alpes. Clermont-Ferrand, le 1er août 2024 Pour la Directrice générale et par délégation, Le directeur délégué régulation de l’offre de soins hospitalière Signé : Jean SCHWEYER Arrêté n°2024-17-0279 Portant autorisation de réguler temporairement l’accès aux urgences du Centre Hospitalier Alpes Léman La Directrice générale de l’Agence Régionale de Santé d’Auvergne - Rhône-Alpes Vu le code de la santé publique, notamment ses articles L. 1432-2, L. [...] de la santé publique, notamment ses articles L. 1432-2, L. 6122-1, L. 6122-8, R. 6122-25, R. 6122-41, R. 6123-1 à R. 6123-32-11 ; Vu l’arrêté du 2 juillet 2024 relatif à la régulation temporaire de l’accès aux urgences ; Vu le courrier du 21 mars 2016 portant renouvellement de l’autorisation de médecine d’urgence du Centre Hospitalier Alpes Léman ; Considérant que tout établissement de santé autorisé à exercer la médecine d’urgence est tenu d’accueillir en permanence dans la structure des urgences toute personne qui s’y présente en situation d’urgence ou qui lui est adressé, notamment par le SAMU ; Considérant qu’aux termes de l’article R. 6123-18-2 du Code de la santé publique : « A titre temporaire et lorsque les circonstances locales le justifient, les établissements disposant d'une [...] des urgences, ARRÊTE Article 1er : A compter du 2 juillet 2024 et jusqu’au 2 septembre 2024, le Centre Hospitalier Alpes Léman est autorisé à réguler l’accès à sa structure des urgences entre 18h et 8h. Article 2 : L’accès à la structure des urgences s’opérera par une régulation préalable après appel au SAMU-Centre 15. L'organisation mise en œuvre à l'entrée de la structure des urgences concernée comporte un accueil physique par un professionnel de santé ou par personne titulaire de l’attestation de formation aux gestes et soins d’urgence (AFGSU). Et La régulation s’opérera par une orientation préalable, en amont de l'accueil du patient et de la prise en charge définis à l'article R. 6123-19, effectuée par un auxiliaire médical de la structure qui met en œuvre des protocoles d'orientation", 'score': 0.79484755, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/bourgogne-franche-comte/irecontenu/telechargement/127841/942996/file/recueil-bfc-2025-076-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N°BFC ...', 'content': 'entreprises de transports sanitaires ; • Reçoit le bilan clinique et indique à l’équipage ambulancier les actions à effectuer en fonction de l’état du patient ; • Indique le lieu d’adressage/destination. 2.2. Sanctions en cas de manquement aux obligations Tout manquement aux obligations règlementaires dans le cadre de la garde et du cahier des charges peut faire l’objet d’une décision de retrait, temporaire ou définitive, d’autorisation de mise en service et/ou d’agrément voire de sanctions judiciaires. Les activités de garde et de transports sanitaires urgents sont soumises aux mêmes règles concernant les véhicules que l’activité de transport sanitaire non spécialisée. ARS Bourgogne Franche-Comté - BFC-2024-10-18-00022 - Arrêté ARS BFC DCPT 2024-59 Organisation de la garde et réponse à [...] robinet permettant un débit d’eau d’au moins 15 l / min, raccord rapide optionnel Insufflateurs manuels avec masques et canules pour tous les âges Embout de ventilation bouche à masque avec entrée oxygène Dispositif portable, manuel, d’aspiration de mucosités Equipements de diagnostic Appareil à tension manuel, taille de serrage 10 cm-66 cm Appareil à tension automatique de type doppler, 10 cm-66 cm Optionnel Oxymètre Optionnel Stéthoscope Optionnel Thermomètre, mesures minimales : 28° C-42° C Optionnel Dispositif pour doser le sucre dans le sang Optionnel ARS Bourgogne Franche-Comté - BFC-2024-10-18-00022 - Arrêté ARS BFC DCPT 2024-59 Organisation de la garde et réponse à la demande de transports sanitaires urgents dans le département du Jura 39 34 Médicaments Un support soluté [...] 2024-59 Organisation de la garde et réponse à la demande de transports sanitaires urgents dans le département du Jura 55 50 ARS Bourgogne Franche-Comté - BFC-2024-10-18-00022 - Arrêté ARS BFC DCPT 2024-59 Organisation de la garde et réponse à la demande de transports sanitaires urgents dans le département du Jura 56 51 ARS Bourgogne Franche-Comté - BFC-2024-10-18-00022 - Arrêté ARS BFC DCPT 2024-59 Organisation de la garde et réponse à la demande de transports sanitaires urgents dans le département du Jura 57 52 ARS Bourgogne Franche-Comté - BFC-2024-10-18-00022 - Arrêté ARS BFC DCPT 2024-59 Organisation de la garde et réponse à la demande de transports sanitaires urgents dans le département du Jura 58 ARS Bourgogne Franche-Comté BFC-2025-05-09-00002 25-17-0291 Arrêté portant autorisation', 'score': 0.39110413, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/auvergne-rhone-alpes/irecontenu/telechargement/134159/983040/file/20251231_recueil-84-2025-368-recueil-des-actes-administratifs-11.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS N° 84-2025-368 ...', 'content': "Article 3 : A compter du 1er janvier 2026, la dotation provisoire du Centre de Soins, d'Accompagnement ... Bourgoin-Jallieu et gérée par l'entité", 'score': 0.3546794, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/irecontenu/telechargement/132991/975666/file/20251126_n%C2%B0-1_recueil-84-2025-332-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "6 : La personne désignée par la Directrice Générale de l’Agence Régionale de Santé Auvergne-Rhône-Alpes est chargée de l’exécution du présent arrêté. Le 23/10/2025 Pour La Directrice Générale de l'Agence Régionale de Santé Auvergne-Rhône-Alpes, et par délégation, Pour le Directeur de l'autonomie, La responsable du pôle qualité Signé, Marguerite POUZET Agence Régionale de Santé Auvergne-Rhône-Alpes, 241 rue Garibaldi CS 93383 – 69418 - LYON CEDEX 03 1/2 Arrêté modificatif annuel FIR n° 2025-15-0069 attribuant des crédits FIR au titre de l’année 2025 la Directrice Générale de l'Agence Régionale de Santé Auvergne-Rhône-Alpes Bénéficiaire : CENTRE HOSPITALIER PIERRE OUDOT 30 AVENUE DU MEDIPOLE 38300 BOURGOIN JALLIEU SIRET - 26380006200238 Code interne - 042791 Vu le code de la santé publique, [...] et la sécurité de l'offre sanitaire et médico-sociale ». Le versement de cette subvention s’effectuera par 12ème. Agence Régionale de Santé Auvergne-Rhône-Alpes, 241 rue Garibaldi CS 93383 – 69418 - LYON CEDEX 03 2/2 Article 4 : A compter du 1er janvier 2026, dans l’attente de la fixation du montant des crédits FIR pour l'année 2026, des acomptes mensuels égaux à un douzième du montant des crédits FIR pour 2025 seront versés à l’établissement : • Base de calcul pour la mesure « MI2-4-20 : équipes mobiles d'hygiène » : 275 713,00 euros, soit un douzième correspondant à 22 976,08 euros. Soit un montant total de 22 976,08 euros. Article 5 : Le présent arrêté peut faire l'objet d'un recours devant le tribunal administratif dans le délai de deux mois à compter de sa notification. Article 6 : [...] de santé coordonnés ainsi que la qualité et la sécurité de l'offre sanitaire et médico-sociale ». Le versement de cette subvention s’effectuera par 12ème. Agence Régionale de Santé Auvergne-Rhône-Alpes, 241 rue Garibaldi CS 93383 – 69418 - LYON CEDEX 03 2/2 Article 4 : A compter du 1er janvier 2026, dans l’attente de la fixation du montant des crédits FIR pour l'année 2026, des acomptes mensuels égaux à un douzième du montant des crédits FIR pour 2025 seront versés à l’établissement : • Base de calcul pour la mesure « MI2-4-20 : équipes mobiles d'hygiène » : 149 320,00 euros, soit un douzième correspondant à 12 443,33 euros. Soit un montant total de 12 443,33 euros. Article 5 : Le présent arrêté peut faire l'objet d'un recours devant le tribunal administratif dans le délai de deux mois à", 'score': 0.3379028, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/irecontenu/telechargement/115321/861066/file/28mars2024_recueil-84-2024-085-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "de l’exécution du présent arrêté, qui sera notifié au demandeur et publié au recueil des actes administratifs de la préfecture de la région Auvergne-Rhône-Alpes et du conseil départemental du Rhône. Fait à Lyon, le 29 décembre 2023 En trois exemplaires originaux Le Directeur général de l’ARS Auvergne-Rhône-Alpes Pour la Directrice Générale et par délégation, Le directeur de l’autonomie Raphaël GLABI Le Président du Département du Rhône Agence régionale de santé Auvergne-Rhône-Alpes CS 93383 - 69418 Lyon cedex 03 04 72 34 74 00 Le Département Rhône 29-31 cours de la Liberté - 69483 Lyon Cedex 03 N° Vert ® : 0 800 869 869 Annexe FINESS Mouvement FINESS : cession d’autorisation Ancienne entité juridique SAS LES OPALINES CHARNAY Adresse Bayère - 69380 CHARNAY N° FINESS EJ 69 002 899 8 Statut [...] et des familles, dans les conditions prévues à l’article L.313-5 du même code. Article 4 : Les caractéristiques de la présente décision sont enregistrées au Fichier national des établissements sanitaires et sociaux (FINESS) comme indiqué dans l’annexe jointe. Article 5 : Tout changement important dans l’activité, l’installation, l’organisation, la direction ou le fonctionnement de l’établissement par rapport aux caractéristiques prises en considération pour son autorisation devra être porté à la connaissance de la Directrice générale de l’Agence régionale de santé Auvergne-Rhône-Alpes selon les termes de l’article L.313-1 du code de l’action sociale et des familles. L’autorisation ne peut être cédée sans son accord. Article 6 : Dans les deux mois suivant sa notification à l’intéressé ou [...] Dans les deux mois suivant sa notification à l’intéressé ou sa publication pour les tiers, la présente décision peut faire l'objet d'un recours gracieux auprès de la Directrice générale de l'Agence régionale de santé Auvergne-Rhône-Alpes ou d'un recours contentieux devant le tribunal administratif compétent. En application du décret n°2018-251 du 6 avril 2018, les particuliers et les personnes morales de droit privé non représentées par un avocat peuvent communiquer avec un Tribunal administratif par la voie de l’application informatique « Télérecours citoyens » sur le site www.telerecours.fr. Article 7 : Le Directeur de la délégation départementale de l’Isère de l'Agence régionale de santé Auvergne-Rhône-Alpes est chargé de l'exécution du présent arrêté, qui sera notifié au demandeur et", 'score': 0.32778713, 'raw_content': None}, {'url': 'https://www.prefectures-regions.gouv.fr/auvergne-rhone-alpes/irecontenu/telechargement/129001/950492/file/20250630_recueil-84-2025-178-recueil-des-actes-administratifs-special.pdf', 'title': 'RECUEIL DES ACTES ADMINISTRATIFS SPÉCIAL N° 84- ...', 'content': "services du Département sont chargés, chacun en ce qui le concerne, de l’exécution du présent arrêté qui sera publié au recueil des actes administratifs de la préfecture de la région Auvergne-Rhône-Alpes et du site internet du département de la Loire. Fait à Lyon, le 20/06/2025 La Directrice générale de l’ARS Auvergne-Rhône-Alpes P/La Directrice Générale et par délégation, Le directeur de l’autonomie Raphaël GLABI Le Président du Département de la Loire Georges ZIEGLER Agence régionale de santé Auvergne-Rhône-Alpes CS 93383 - 69418 Lyon cedex 03 04 72 34 74 00 Le Département de la Loire 2 rue Charles de Gaulle – 42000 Saint-Etienne 04 77 48 42 42 Annexe FINESS Mouvements FINESS : Mise en œuvre d’un dispositif expérimental Entité juridique : EURECAH Adresse : Allée Lavoisier - 42350 LA [...] les deux mois suivant sa notification ou sa publication, la présente décision peut faire l'objet d'un recours gracieux auprès de la Directrice générale de l'Agence régionale de santé Auvergne-Rhône-Alpes et/ou du Président du Département de la Haute-Savoie, ou d'un recours contentieux devant le tribunal administratif compétent. En application du décret n°2018-251 du 6 avril 2018, les particuliers et les personnes morales de droit privé non représentées par un avocat peuvent communiquer avec un Tribunal administratif par la voie de l’application informatique « Télérecours citoyens » sur le site www.telerecours.fr. Article 7 : Le Directeur départemental de la Haute-Savoie de l’Agence régionale de santé Auvergne-Rhône-Alpes ainsi que le Président du Département de la Haute-Savoie sont [...] que le Président du Département de la Haute-Savoie sont chargés, chacun en ce qui le concerne, de l’exécution du présent arrêté, qui sera notifié au demandeur et publié au recueil des actes administratifs de la préfecture de la région Auvergne-Rhône-Alpes et du Département de la Haute-Savoie. Fait à Annecy, le 26/06/2025 La Directrice générale de l’ARS Auvergne-Rhône-Alpes P/La Directrice Générale et par délégation, Le directeur de l’autonomie Raphaël GLABI Le Président du Conseil départemental de la Haute-Savoie Agence régionale de santé Auvergne-Rhône-Alpes CS 93383 - 69418 Lyon cedex 03 04 72 34 74 00 Le Département de la Haute-Savoie CS 32444 – 74041 Annecy cedex 04 50 33 50 00 Annexe FINESS Mouvements FINESS : Renouvellement de l’autorisation de fonctionnement et changement de", 'score': 0.27455238, 'raw_content': None}, {'url': 'https://www.auvergne-rhone-alpes.ars.sante.fr/media/141546/download?inline', 'title': 'communiqué de presse', 'content': "L'ensemble des salariés du service de pédopsychiatrie du Centre Hospitalier auront la possibilité de travailler au sein de l'ESMPI selon des modalités qui seront à définir en lien avec les représentants du personnel, afin de garantir le maintien des compétences d'expertises au profit des patients ainsi que le bassin d'emploi.\nCOMMUNIQUÉ DE PRESSE 09 juillet 2025 Centre Hospitalier Pierre Oudot 30 avenue du Médipôle 38300 Bourgoin-Jallieu +33 (0)4 69 15 70 00 www.ch-bourgoin.fr communication@ghnd.fr PEDOPSYCHIATRIE DU CENTRE HOSPITALIER PIERRE OUDOT : UNE REPRISE NECESSAIRE PAR L’ETABLISSEMENT DE SANTE MENTALE DES PORTES DE L’ISERE (ESMPI) POUR ASSURER LA CONTINUITE DES PRISES EN CHARGE [...] Ces dernières années, le Centre Hospitalier Pierre Oudot à Bourgoin-Jallieu a été conduit à restructurer l'organisation des prises en charge des enfants accueillis en pédopsychiatrie pour faire face au nombre limité de pédopsychiatres sur son territoire. Un dispositif rigoureux d'évaluation et d'orientation des enfants a été mis en place permettant de définir un parcours de soin clair, régulièrement réévalué et planifié. Cette offre est cependant restée fragilisée par la présence d'un seul pédopsychiatre ; ce manque d'effectifs médicaux dans cette spécialité est un problème à la fois national mais aussi local auquel le Centre hospitalier a dû faire face. Le transfert du service de pédopsychiatrie au sein de l’ESMPI : la seule solution possible pour maintenir l’offre de soins Face à un [...] Le scénario retenu, qui est apparu comme le seul possible et cohérent avec l’organisation territoriale de la psychiatrie, a été celui de la reprise de l'activité de pédopsychiatrie par l'Etablissement de Santé Mentale des Portes de l'Isère (ESMPI). Celui-ci propose déjà une offre de psychiatrie adulte sur l’ensemble du Nord-Isère, y compris donc le territoire de Bourgoin-Jallieu. L’ESMPI assure également l’activité de pédopsychiatrie sur le territoire de Vienne. Un scénario retenu pour permettre le maintien d'une offre de proximité pour les enfants et leurs familles, sans rupture des accompagnements En lien avec l’Agence régionale de santé Auvergne-Rhône-Alpes, les directions des deux établissements organisent cette transition qui devra garantir pour les patients et leurs familles la", 'score': 0.2104582, 'raw_content': None}, {'url': 'https://www.auvergne-rhone-alpes.ars.sante.fr/pedopsychiatrie-du-centre-hospitalier-pierre-oudot-une-reprise-necessaire-par-letablissement-de', 'title': 'Pédopsychiatrie du Centre hospitalier Pierre-Oudot', 'content': "Celui-ci propose déjà une offre de psychiatrie adulte sur l'ensemble du Nord-Isère, y compris donc le territoire de Bourgoin-Jallieu. L'ESMPI", 'score': 0.07512905, 'raw_content': None}]}

In [1]:
import json
from vllm import LLM, SamplingParams


/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    gpu_memory_utilization=0.55,  # conservador
    max_model_len=2048,           # CLAVE
)

params = SamplingParams(
    temperature=0.7,
    max_tokens=128,
)

out = llm.generate(
    ["Explique en une phrase ce qu’est une urgence médicale."],
    params
)

print(out[0].outputs[0].text)

INFO 02-03 17:36:22 [utils.py:261] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.55, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
INFO 02-03 17:36:23 [model.py:541] Resolved architecture: MistralForCausalLM
INFO 02-03 17:36:23 [model.py:1561] Using max model len 2048


2026-02-03 17:36:24,059	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-03 17:36:24 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-03 17:36:24 [vllm.py:624] Asynchronous scheduling is enabled.


[2026-02-03 17:36:24] WARNING utils.py:121: Multiple valid tokenizer files found. Using tokenizer.model.v3.


(EngineCore_DP0 pid=1276438) INFO 02-03 17:36:24 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='mistralai/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='mistralai/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_

(EngineCore_DP0 pid=1276438) Exception in thread Thread-4 (_report_usage_worker):
(EngineCore_DP0 pid=1276438) Traceback (most recent call last):
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
(EngineCore_DP0 pid=1276438)     self.run()
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
(EngineCore_DP0 pid=1276438)     _threading_Thread_run(self)
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/threading.py", line 953, in run
(EngineCore_DP0 pid=1276438)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/usage/usage_lib.py", line 173, in _report_usag

(EngineCore_DP0 pid=1276438) INFO 02-03 17:36:35 [cuda.py:364] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=1276438) ERROR 02-03 17:48:52 [core.py:946] EngineCore failed to start.
(EngineCore_DP0 pid=1276438) ERROR 02-03 17:48:52 [core.py:946] Traceback (most recent call last):
(EngineCore_DP0 pid=1276438) ERROR 02-03 17:48:52 [core.py:946]   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 937, in run_engine_core
(EngineCore_DP0 pid=1276438) ERROR 02-03 17:48:52 [core.py:946]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore_DP0 pid=1276438) ERROR 02-03 17:48:52 [core.py:946]   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 691, in __init__
(EngineCore_DP0 pid=1276438) ERROR 02-03 17:48:52 [co

(EngineCore_DP0 pid=1276438) Process EngineCore_DP0:
(EngineCore_DP0 pid=1276438) Traceback (most recent call last):
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=1276438)     self.run()
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=1276438)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 950, in run_engine_core
(EngineCore_DP0 pid=1276438)     raise e
(EngineCore_DP0 pid=1276438)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 937, in run_engine_core
(EngineCore_DP0 pid=

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}